# Chapter 6b — RAG over Research Papers

Companion code for the **second half of Chapter 6** of *Build an Advanced RAG Application (From Scratch)*.

Same retrieve → augment → generate pipeline as 6a, but a very different corpus: 500 research-paper abstracts from the [DBLP-v10 dataset on Kaggle](https://www.kaggle.com/datasets/nechbamohammed/research-papers-dataset). New things in this notebook:

- **Document chunking** — papers can exceed the embedder's context window.
- **Direct `transformers`** instead of `sentence-transformers` — useful when you want pooling control.
- **Metadata filtering** by year — show only post-2010 papers, etc.
- **Persistent Qdrant** — collection survives across notebook restarts.

> Reusable code: `chunking.py`, `qdrant_helpers.py`.

## 0. Setup

You need three keys in `.env`:

- `OPEN_ROUTER_API_KEY` — for generation
- `KAGGLE_USERNAME` and `KAGGLE_KEY` — for downloading the dataset

Get a Kaggle token at https://www.kaggle.com/settings → *Create New Token*.

In [1]:
import os, sys, csv
sys.path.insert(0, '.')

# Lift the CSV size limit — DBLP rows can be big
csv.field_size_limit(sys.maxsize)

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path('..').resolve() / '.env')

True

## 1. Download the dataset (Kaggle)

`kagglehub` reads `KAGGLE_USERNAME` / `KAGGLE_KEY` from the environment. The dataset is ~150MB.

In [2]:
import kagglehub

path = kagglehub.dataset_download("nechbamohammed/research-papers-dataset")
print("Path:", path)
print(os.listdir(path))

Path: C:\Users\Areej\.cache\kagglehub\datasets\nechbamohammed\research-papers-dataset\versions\1
['dblp-v10.csv']


In [3]:
df = pd.read_csv(Path(path) / "dblp-v10.csv")
print(df.shape)
df.head()

(1000000, 8)


,abstract,authors,n_citation,references,title,venue,year,id
0,"In this paper, a robust 3D triangular mesh wat...","['S. Ben Jabra', 'Ezzeddine Zagrouba']",50,"['09cb2d7d-47d1-4a85-bfe5-faa8221e644b', '10aa...",A new approach of 3D watermarking based on ima...,international symposium on computers and commu...,2008,4ab3735c-80f1-472d-b953-fa0557fed28b
1,We studied an autoassociative neural network w...,"['Joaquín J. Torres', 'Jesús M. Cortés', 'Joaq...",50,"['4017c9d2-9845-4ad2-ad5b-ba65523727c5', 'b118...",Attractor neural networks with activity-depend...,Neurocomputing,2007,4ab39729-af77-46f7-a662-16984fb9c1db
2,It is well-known that Sturmian sequences are t...,"['Genevi eve Paquin', 'Laurent Vuillon']",50,"['1c655ee2-067d-4bc4-b8cc-bc779e9a7f10', '2e4e...",A characterization of balanced episturmian seq...,Electronic Journal of Combinatorics,2007,4ab3a4cf-1d96-4ce5-ab6f-b3e19fc260de
3,One of the fundamental challenges of recognizi...,"['Yaser Sheikh', 'Mumtaz Sheikh', 'Mubarak Shah']",221,"['056116c1-9e7a-4f9b-a918-44eb199e67d6', '05ac...",Exploring the space of a human action,international conference on computer vision,2005,4ab3a98c-3620-47ec-b578-884ecf4a6206
4,This paper generalizes previous optimal upper ...,"['Efraim Laksman', 'Håkan Lennerstad', 'Magnus...",0,"['01a765b8-0cb3-495c-996f-29c36756b435', '5dbc...",Generalized upper bounds on the minimum distan...,Ima Journal of Mathematical Control and Inform...,2015,4ab3b585-82b4-4207-91dd-b6bce7e27c4e


## 2. Prepare a small slice for the demo

Full DBLP is too big to embed in a notebook session. We take 500 papers, drop those without abstracts, and shape each row into a `{page_content, metadata}` dict.

In [4]:
df_small = df.head(500).copy()
df_small = df_small.dropna(subset=["abstract"]).reset_index(drop=True)

documents = [
    {
        "page_content": row["abstract"],
        "metadata": {
            "source": row["title"],
            "authors": row["authors"],
            "year": row["year"],
            "venue": row["venue"],
            "paper_id": row["id"],
        },
    }
    for _, row in df_small.iterrows()
]
print(f"Documents: {len(documents)}")

Documents: 479


## 3. Chunk the documents

Most abstracts fit comfortably in one chunk, but the same code handles longer papers. See `chunking.py`.

In [5]:
from chunking import simple_recursive_split

chunks = []
for doc in documents:
    chunks.extend(simple_recursive_split(doc, chunk_size=4000, chunk_overlap=200))

print(f"Total chunks: {len(chunks)}")
print(f"First chunk length: {len(chunks[0]['page_content'])}")
chunks[0]

Total chunks: 479
First chunk length: 511


{'page_content': 'In this paper, a robust 3D triangular mesh watermarking algorithm based on 3D segmentation is proposed. In this algorithm three classes of watermarking are combined. First, we segment the original image to many different regions. Then we mark every type of region with the corresponding algorithm based on their curvature value. The experiments show that our watermarking is robust against numerous attacks including RST transformations, smoothing, additive random noise, cropping, simplification and remeshing.',
 'metadata': {'source': 'A new approach of 3D watermarking based on image segmentation',
  'authors': "['S. Ben Jabra', 'Ezzeddine Zagrouba']",
  'year': 2008,
  'venue': 'international symposium on computers and communications',
  'paper_id': '4ab3735c-80f1-472d-b953-fa0557fed28b'}}

## 4. Generate embeddings (transformers, mean-pooled)

We load the model with `transformers` directly so we can see exactly how the pooling works.

In [6]:
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True).to(device)

def embed(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = text_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu()[0].numpy()

# Smoke test
v = embed("autoassociative neural network with dynamic synapses")
print(f"Embedding dim: {v.shape}")

Device: cpu


<All keys matched successfully>


Embedding dim: (768,)


In [7]:
embeddings = [embed(c["page_content"]) for c in chunks]
print(f"Embeddings: {len(embeddings)} x {embeddings[0].shape}")

Embeddings: 479 x (768,)


## 5. Index in Qdrant

Same helpers as 6a; different collection name.

In [8]:
from qdrant_helpers import make_in_memory_client, reset_collection, upsert_text_chunks, query_papers
from qdrant_client import models

qdrant = make_in_memory_client()
COLLECTION = "research_collection"
VEC_SIZE = embeddings[0].shape[0]
reset_collection(qdrant, COLLECTION, vector_size=VEC_SIZE)
upsert_text_chunks(qdrant, COLLECTION, chunks, embeddings)
print(f"Upserted {len(chunks)} chunks into '{COLLECTION}'.")

Upserted 479 chunks into 'research_collection'.


## 6. Query

In [9]:
hits = query_papers(
    "an autoassociative neural network with dynamic synapses",
    embed,
    qdrant,
    collection_name=COLLECTION,
    limit=3,
)
for i, h in enumerate(hits, 1):
    md_meta = h.payload["metadata"]
    print(f"{i}. {md_meta['source']}  ({md_meta['year']}, {md_meta['venue']})")
    print(f"   {h.payload['content'][:240]}...\n")

1. Attractor neural networks with activity-dependent synapses: The role of synaptic facilitation  (2007, Neurocomputing)
   We studied an autoassociative neural network with dynamic synapses which include a facilitating mechanism. We have developed a general mean-field framework to study the relevance of the different parameters defining the dynamics of the syna...

2. Transient Signal Detection with Neural Networks: The Search for the Desired Signal  (1993, neural information processing systems)
   Matched filtering has been one of the most powerful techniques employed for transient detection. Here we will show that a dynamic neural network outperforms the conventional approach. When the artificial neural network (ANN) is trained with...

3. Prerequesites for symbiotic brain-machine interfaces  (2009, systems, man and cybernetics)
   Recent advancements in the neuroscience and engineering of Brain-Machine Interfaces are providing a blueprint for how new co-adaptive designs based on re

### With a year filter

Qdrant supports server-side filters on payload fields — pass them at query time.

In [10]:
query_vec = embed("an autoassociative neural network with dynamic synapses")
hits = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_vec.tolist(),
    query_filter=models.Filter(
        must=[models.FieldCondition(key="metadata.year", range=models.Range(gte=2000))]
    ),
    limit=3,
).points

for i, h in enumerate(hits, 1):
    m = h.payload["metadata"]
    print(f"{i}. {m['source']}  (year={m['year']})  score={h.score:.3f}")

1. Attractor neural networks with activity-dependent synapses: The role of synaptic facilitation  (year=2007)  score=0.825
2. Prerequesites for symbiotic brain-machine interfaces  (year=2009)  score=0.638
3. Robust fuzzy and recurrent neural network motion control among dynamic obstacles for robot manipulators  (year=2000)  score=0.637


## 7. Full RAG over papers

In [11]:
from openai import OpenAI

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPEN_ROUTER_API_KEY"))

def query_qdrant_for_rag(query, limit=5):
    hits = query_papers(query, embed, qdrant, collection_name=COLLECTION, limit=limit)
    return [
        {
            "source_id": i + 1,
            "content": h.payload["content"],
            "metadata": h.payload["metadata"],
        }
        for i, h in enumerate(hits)
    ]

def generate_paper_answer(query, llm_model="qwen/qwen3-8b"):
    sources = query_qdrant_for_rag(query)
    prompt = f'''Based on the following query, generate a comprehensive answer.
Cite inline as [1][2] and mention authors, paper titles, and venues. Be precise.

Query: "{query}"

Sources:
{sources}

Return in Markdown format.'''
    stream = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    out = ""
    for chunk in stream:
        d = chunk.choices[0].delta.content
        if d:
            print(d, end="", flush=True)
            out += d
    print()
    return out, sources

In [12]:
answer, sources = generate_paper_answer("How do autoassociative neural networks with dynamic synapses store information?")

Autoassociative neural networks with dynamic synapses store information through mechanisms that leverage synaptic dynamics to enhance retrieval efficiency and adaptability. A key study by Torres et al. (2007) investigates such networks, emphasizing the role of synaptic facilitation—a dynamic synapse mechanism that strengthens synaptic connections during repeated activation [1]. The authors develop a mean-field framework to analyze how parameters governing synaptic dynamics influence the network's collective behavior. This framework reveals that the network exhibits distinct operational phases: a retrieval phase, where stored patterns are accessed; an oscillatory regime, where activity alternates between stored patterns; and a non-retrieval phase, where patterns are not accessible [1]. 

In the oscillatory phase, the network continuously switches between stored patterns, enabling rapid and error-reduced access to information when facilitation is strong. This is contrasted with mechanism

## Wrap

You now have the same RAG pattern instantiated on two completely different corpora — hotel reviews and research papers. The pieces (embedder, vector DB, retriever, prompt template, LLM) are interchangeable; the architecture is the same.